# Customer Support Ticket Intelligence Platform
## Phase 5: Simple RNN — From Bag of Words to Sequential Processing

**Objective**: Build our first *sequence-aware* deep learning model (MODEL-v2: Simple RNN) and compare it head-to-head against the sequence-agnostic BoE baseline from Phase 4.

**Key Question**: Does processing tokens in order (RNN) outperform treating them as an unordered bag (BoE)?

**What we'll learn**:
1. How RNNs process tokens sequentially with a hidden state
2. Why `pack_padded_sequence` is essential for correct RNN training
3. How gradient clipping prevents training instability
4. Head-to-head comparison: BoE vs Simple RNN

**Experiment**: EXP-05 — Simple RNN (50d embedding, 64h hidden)


In [ ]:
# ==============================================================================
# ENVIRONMENT SETUP & DEVICE CONFIGURATION BOOTSTRAP CELL
# ==============================================================================
import os
import sys
from pathlib import Path

# 1. Detect Environment
IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    print("Detected Environment: Google Colab")
    
    # Mount Google Drive if requested (uncomment if needed)
    # from google.colab import drive
    # drive.mount('/content/drive')
    
    # Clone the repository if not present
    repo_name = "customer-support-ticket-intelligence-platform"
    repo_url = f"https://github.com/ikartiksavaliya/{repo_name}.git"
    target_dir = f"/content/{repo_name}"
    
    if not os.path.exists(target_dir):
        print(f"Cloning repository {repo_url}...")
        import subprocess
        subprocess.run(["git", "clone", repo_url, target_dir], check=True)
    
    # Change working directory to the repository root
    os.chdir(target_dir)
    
    # Add project root to sys.path
    if target_dir not in sys.path:
        sys.path.insert(0, target_dir)
        
    # Install requirements
    print("Installing requirements.txt and package in editable mode...")
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)
else:
    print("Detected Environment: Local Machine / VS Code")
    # Add project root to sys.path by climbing up directories
    ROOT = Path.cwd()
    while ROOT != ROOT.parent and not (ROOT / "src").exists():
        ROOT = ROOT.parent
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))
    os.chdir(str(ROOT))

# 2. Verify Imports & Configure Device
import torch
try:
    from src.utils import get_device, set_seed
    from src.preprocessing import clean_text
    from src.vocabulary import Vocabulary
    from src.dataset import TicketDataset
    print("✅ Project modules successfully imported!")
except ImportError as e:
    print(f"❌ Failed to import project modules: {e}")
    raise e

# Initialize seed for reproducibility
set_seed(42)

# Get compute device
device, device_type, device_name = get_device()


---
### Section 1: Environment Setup & Data Loading


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("Current device:", torch.cuda.current_device())
    print("Device name:", torch.cuda.get_device_name(0))

In [ ]:
import sys, os, json, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


from src.utils import set_seed, get_device, compute_class_weights
from src.preprocessing import clean_text, tokenize
from src.vocabulary import Vocabulary
from src.dataset import TicketDataset
from src.models import BagOfEmbeddings, SimpleRNNClassifier
from src.training import train_model, evaluate
from src.evaluation import (
    compute_metrics, print_classification_report,
    plot_confusion_matrix, plot_training_curves
)

# Reproducibility
set_seed(42)
device, _, _ = get_device()
print(f"PyTorch version: {torch.__version__}")


In [ ]:
# Load pre-built artifacts
vocab = Vocabulary.load("../outputs/vocab.json")
with open("../outputs/label_encoder.json", "r") as f:
    label_encoder = json.load(f)

idx2label = {v: k for k, v in label_encoder.items()}
label_names = [idx2label[i] for i in range(len(label_encoder))]

# Load data splits
train_df = pd.read_csv("../data/splits/train.csv")
val_df   = pd.read_csv("../data/splits/val.csv")
test_df  = pd.read_csv("../data/splits/test.csv")

print(f"Vocabulary size : {len(vocab):,}")
print(f"Number of classes: {len(label_encoder)}")
print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")


---
### Section 2: Create DataLoaders with Sequence Lengths

For RNN models, we need the **actual sequence lengths** (before padding) to use `pack_padded_sequence`. We enable this with `return_lengths=True` in our `TicketDataset`.


In [ ]:
MAX_LEN = 256
BATCH_SIZE = 64

# RNN datasets return 3-tuples: (input_ids, label, seq_len)
train_ds = TicketDataset(train_df, vocab, label_encoder, max_len=MAX_LEN, return_lengths=True)
val_ds   = TicketDataset(val_df,   vocab, label_encoder, max_len=MAX_LEN, return_lengths=True)
test_ds  = TicketDataset(test_df,  vocab, label_encoder, max_len=MAX_LEN, return_lengths=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

# Verify 3-tuple output
sample_ids, sample_labels, sample_lens = next(iter(train_loader))
print(f"Batch input_ids shape : {sample_ids.shape}")
print(f"Batch labels shape    : {sample_labels.shape}")
print(f"Batch lengths shape   : {sample_lens.shape}")
print(f"Sequence lengths (first 10): {sample_lens[:10].tolist()}")
print(f"Min length: {sample_lens.min().item()}, Max length: {sample_lens.max().item()}")


---
### Section 3: Theory — Recurrent Neural Networks (RNN)

#### From Bag to Sequence: Why Order Matters

The Bag of Embeddings model (Phase 4) treats each document as an unordered collection of word vectors:

$$\text{BoE}(\text{doc}) = \frac{1}{T} \sum_{t=1}^{T} e_{w_t}$$

This means "payment was not received" and "received was payment not" produce **identical** representations. For financial complaint classification, this is a real problem — the *sequence* of events matters.

#### The RNN Solution: Sequential Processing with Memory

An RNN processes tokens **one at a time**, maintaining a **hidden state** $h_t$ that summarises all tokens seen so far:

$$h_t = \tanh(W_{hh} \cdot h_{t-1} + W_{xh} \cdot x_t + b_h)$$

Where:
- $h_{t-1} \in \mathbb{R}^{h}$ is the previous hidden state (the "memory")
- $x_t \in \mathbb{R}^{d}$ is the current input embedding
- $W_{hh} \in \mathbb{R}^{h \times h}$ is the hidden-to-hidden weight matrix (same at every step!)
- $W_{xh} \in \mathbb{R}^{h \times d}$ is the input-to-hidden weight matrix
- $\tanh$ squashes the output to $[-1, 1]$

The key insight: **$W_{hh}$ and $W_{xh}$ are shared across all time steps**. The RNN learns *one* set of transition rules that it applies at every position — this is what gives it the ability to generalise to different sequence lengths.

#### Unrolled View

```
x_1    x_2    x_3    ...   x_T
 |      |      |            |
 v      v      v            v
[RNN] → [RNN] → [RNN] → ... → [RNN] → h_T → Linear → logits
  ↑       ↑       ↑              ↑
 h_0     h_1     h_2           h_{T-1}
```

The final hidden state $h_T$ is a compressed summary of the entire input sequence, which we feed to the linear classification head.

#### Computational Complexity
- **Per timestep**: $O(h^2 + h \cdot d)$ — dominated by the $W_{hh} \cdot h_{t-1}$ multiplication
- **Per sequence**: $O(T \cdot (h^2 + h \cdot d))$ — **sequential**, cannot be parallelised
- This is why RNNs are slower than BoE, which processes all tokens in parallel


---
### Section 4: Theory — Why `pack_padded_sequence` is Critical

#### The Problem with Padding

Our sequences are padded to `max_len=256` with zeros. Without packing, the RNN processes **all 256 positions** — including the PAD tokens. After 200+ steps through zero embeddings, the hidden state is dominated by the bias term and $\tanh$ saturation:

```
Real tokens (50)          PAD tokens (206)
h_1 → h_2 → ... → h_50 → h_51 → ... → h_256
  ↑ rich information         ↑ garbage — hidden state washed out
```

The hidden state at position 256 no longer represents the text — it represents 206 steps of processing zeros through $\tanh(W_{hh} \cdot h + b)$.

#### The Solution: `pack_padded_sequence`

`torch.nn.utils.rnn.pack_padded_sequence` tells the RNN to **stop processing at the actual sequence end**:

```python
packed = pack_padded_sequence(embedded, lengths, batch_first=True, enforce_sorted=False)
_, h_n = self.rnn(packed, h_0)
# h_n now contains the hidden state at the LAST REAL TOKEN
```

The returned `h_n` corresponds to the hidden state at token 50 (the last real token), not token 256 (the last PAD token).


In [ ]:
# Demonstrate the difference: with vs without packing
set_seed(42)
demo_model = SimpleRNNClassifier(vocab_size=len(vocab), embedding_dim=50, hidden_dim=64, num_classes=len(label_encoder)).to(device)

# Create a sample with lots of padding
demo_ids = sample_ids[0:1].to(device)     # single sample
demo_len = sample_lens[0:1]                # its actual length

# Forward with packing (correct)
out_packed = demo_model(demo_ids, demo_len)

# Forward without packing (incorrect — processes PAD tokens)
out_no_pack = demo_model(demo_ids, None)

print(f"Sequence length: {demo_len.item()} out of {MAX_LEN}")
print(f"Output WITH packing    : {out_packed[0, :5].detach().cpu().tolist()}")
print(f"Output WITHOUT packing : {out_no_pack[0, :5].detach().cpu().tolist()}")
print(f"Outputs differ: {not torch.allclose(out_packed, out_no_pack, atol=1e-4)}")
print()
print("⚠️  Without packing, the RNN processes 200+ PAD tokens,")
print("    washing out the real text signal from the hidden state!")


---
### Section 5: Build MODEL-v2 — Simple RNN Classifier


In [ ]:
set_seed(42)
EMBEDDING_DIM = 50
HIDDEN_DIM = 64

model = SimpleRNNClassifier(
    vocab_size=len(vocab),
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=len(label_encoder),
).to(device)

print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# Compare with BoE
boe_params = 23262 * 50 + 50 * 18 + 18  # Embedding + Linear
rnn_params = sum(p.numel() for p in model.parameters())
print(f"\nBoE (50d) params : {boe_params:,}")
print(f"RNN (50d, 64h)   : {rnn_params:,}")
print(f"RNN overhead     : +{rnn_params - boe_params:,} params ({(rnn_params/boe_params - 1)*100:.1f}% more)")


---
### Section 6: Training Setup — Custom Loop for RNN

Since our RNN model requires `seq_lengths` as a second input, we need a slightly modified training loop that unpacks the 3-tuple from our DataLoader and passes lengths to the model.


In [ ]:
# Class weights for imbalanced data
class_weights = compute_class_weights(label_encoder, train_df)
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

# RNN-aware training functions
def train_one_epoch_rnn(model, dataloader, optimizer, criterion, device, max_grad_norm=1.0):
    """Train one epoch with pack_padded_sequence support."""
    model.train()
    running_loss = 0.0
    num_batches = 0
    all_preds, all_labels = [], []
    
    for input_ids, labels, seq_lens in dataloader:
        input_ids = input_ids.to(device)
        labels = labels.to(device)
        
        logits = model(input_ids, seq_lens)
        loss = criterion(logits, labels)
        
        optimizer.zero_grad()
        loss.backward()
        
        # Gradient clipping — essential for RNN stability
        if max_grad_norm is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        
        optimizer.step()
        
        running_loss += loss.item()
        num_batches += 1
        all_preds.extend(logits.argmax(dim=1).cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
    
    return running_loss / max(num_batches, 1), all_preds, all_labels


@torch.no_grad()
def evaluate_rnn(model, dataloader, criterion, device):
    """Evaluate with pack_padded_sequence support."""
    model.eval()
    running_loss = 0.0
    num_batches = 0
    all_preds, all_labels = [], []
    
    for input_ids, labels, seq_lens in dataloader:
        input_ids = input_ids.to(device)
        labels = labels.to(device)
        
        logits = model(input_ids, seq_lens)
        loss = criterion(logits, labels)
        
        running_loss += loss.item()
        num_batches += 1
        all_preds.extend(logits.argmax(dim=1).cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
    
    return running_loss / max(num_batches, 1), all_preds, all_labels

print("✅ RNN training loop defined with:")
print("   - pack_padded_sequence support (3-tuple unpacking)")
print("   - Gradient clipping (max_norm=1.0)")


---
### Section 7: Train Simple RNN — EXP-05

Training with Adam (lr=1e-3), gradient clipping (max_norm=1.0), 10 epochs.


In [ ]:
set_seed(42)
EPOCHS = 10
MAX_GRAD_NORM = 1.0

model = SimpleRNNClassifier(
    vocab_size=len(vocab),
    embedding_dim=50,
    hidden_dim=64,
    num_classes=len(label_encoder),
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print(f"Training SimpleRNN for {EPOCHS} epochs...")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Gradient clipping: max_norm={MAX_GRAD_NORM}")
print(f"Device: {device}")
print()

from sklearn.metrics import f1_score
from pathlib import Path

history = {
    "train_loss": [], "val_loss": [], "val_f1": [], "epoch_time": []
}
best_val_f1 = 0.0

print(f"{'Epoch':>7} | {'Train Loss':>11} | {'Val Loss':>9} | {'Val F1':>7} | {'Time':>6}")
print("-" * 55)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    
    train_loss, _, _ = train_one_epoch_rnn(
        model, train_loader, optimizer, criterion, device, max_grad_norm=MAX_GRAD_NORM
    )
    val_loss, val_preds, val_labels = evaluate_rnn(
        model, val_loader, criterion, device
    )
    val_f1 = f1_score(val_labels, val_preds, average="macro", zero_division=0)
    epoch_time = time.time() - t0
    
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_f1"].append(val_f1)
    history["epoch_time"].append(epoch_time)
    
    improved = ""
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        improved = " ✓"
        Path("../outputs").mkdir(exist_ok=True)
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_f1": val_f1, "val_loss": val_loss,
        }, "../outputs/rnn_best.pt")
    
    print(f"  [{epoch:>2}/{EPOCHS}] | {train_loss:>11.4f} | {val_loss:>9.4f} | {val_f1:>6.4f} | {epoch_time:>5.1f}s{improved}")

print("-" * 55)
print(f"Best Val F1: {best_val_f1:.4f}")


---
### Section 8: Evaluate Simple RNN — Classification Report & Curves


In [ ]:
# Training curves
plot_training_curves(history, title="Simple RNN Training Curves", save_path="../outputs/rnn_curves.png")


In [ ]:
# Validation metrics
val_loss, val_preds, val_labels = evaluate_rnn(model, val_loader, criterion, device)
metrics_rnn = compute_metrics(val_labels, val_preds)

print("\nValidation Metrics (Simple RNN):")
for k, v in metrics_rnn.items():
    print(f"  {k:20s}: {v:.4f}")


In [ ]:
# Per-class classification report
print("\nPer-Class Classification Report:")
print_classification_report(val_labels, val_preds, label_names)


In [ ]:
# Confusion matrix
plot_confusion_matrix(val_labels, val_preds, label_names, save_path="../outputs/rnn_confusion.png")


---
### Section 9: Head-to-Head Comparison — BoE vs Simple RNN

The fundamental question of Phase 5: **Does sequential processing help?**

We compare MODEL-v1 (Bag of Embeddings) against MODEL-v2 (Simple RNN) on identical data.


In [ ]:
# Load BoE results from Phase 4
boe_checkpoint = torch.load("../outputs/boe_50d_best.pt", map_location=device, weights_only=True)

boe_model = BagOfEmbeddings(
    vocab_size=len(vocab), embedding_dim=50, num_classes=len(label_encoder)
).to(device)
boe_model.load_state_dict(boe_checkpoint["model_state_dict"])
boe_model.eval()

# Evaluate BoE on validation set (using 2-tuple loader)
val_ds_boe = TicketDataset(val_df, vocab, label_encoder, max_len=MAX_LEN, return_lengths=False)
val_loader_boe = DataLoader(val_ds_boe, batch_size=BATCH_SIZE, shuffle=False)

from src.training import evaluate as evaluate_boe
boe_val_loss, boe_preds, boe_labels = evaluate_boe(boe_model, val_loader_boe, criterion, device)
metrics_boe = compute_metrics(boe_labels, boe_preds)

print("Head-to-Head Comparison:")
print("=" * 60)
print(f"{'Metric':<25} {'BoE (50d)':>15} {'RNN (50d, 64h)':>15}")
print("-" * 60)
for key in ['accuracy', 'f1_macro', 'precision_macro', 'recall_macro']:
    print(f"  {key:<23} {metrics_boe[key]:>14.4f} {metrics_rnn[key]:>14.4f}")

boe_params = sum(p.numel() for p in boe_model.parameters())
rnn_params = sum(p.numel() for p in model.parameters())
print(f"  {'parameters':<23} {boe_params:>14,} {rnn_params:>14,}")

# Which model wins?
f1_diff = metrics_rnn['f1_macro'] - metrics_boe['f1_macro']
winner = "Simple RNN" if f1_diff > 0 else "BoE"
print(f"\n{'='*60}")
print(f"Winner: {winner} (F1 diff: {f1_diff:+.4f})")


In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# F1 comparison
models = ['BoE (50d)', 'Simple RNN\n(50d, 64h)']
f1_scores = [metrics_boe['f1_macro'], metrics_rnn['f1_macro']]
colors = ['#2196F3', '#FF9800']

axes[0].bar(models, f1_scores, color=colors, width=0.5, edgecolor='white')
axes[0].set_ylabel("Macro F1 Score")
axes[0].set_title("F1 Score: BoE vs Simple RNN")
axes[0].set_ylim(0, max(f1_scores) * 1.2)
for i, v in enumerate(f1_scores):
    axes[0].text(i, v + 0.01, f"{v:.4f}", ha='center', fontsize=12, fontweight='bold')

# Per-class F1 comparison (top 8 classes by support)
from sklearn.metrics import f1_score as f1_per_class
boe_f1_per_class = f1_per_class(boe_labels, boe_preds, average=None, zero_division=0)
rnn_f1_per_class = f1_per_class(val_labels, val_preds, average=None, zero_division=0)

# Sort by support (class frequency)
class_counts = val_df['category'].value_counts()
top_classes = [(label_encoder[name], name) for name in class_counts.index[:8]]

x = np.arange(len(top_classes))
width = 0.35
axes[1].bar(x - width/2, [boe_f1_per_class[idx] for idx, _ in top_classes], width, label='BoE', color='#2196F3')
axes[1].bar(x + width/2, [rnn_f1_per_class[idx] for idx, _ in top_classes], width, label='RNN', color='#FF9800')
axes[1].set_ylabel("F1 Score")
axes[1].set_title("Per-Class F1: BoE vs Simple RNN (Top 8 Classes)")
axes[1].set_xticks(x)
axes[1].set_xticklabels([name[:20] for _, name in top_classes], rotation=45, ha='right', fontsize=8)
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle("MODEL-v1 (BoE) vs MODEL-v2 (Simple RNN)", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig("../outputs/boe_vs_rnn_comparison.png", dpi=150, bbox_inches='tight')
plt.show()


---
### Section 10: Key Takeaways & What's Next

#### Findings
1. **Sequential processing** via RNN captures word order that BoE completely ignores.
2. **`pack_padded_sequence`** is essential — without it, PAD tokens wash out the hidden state.
3. **Gradient clipping** (`max_norm=1.0`) prevents training instability from exploding gradients.
4. The RNN has a **sequential bottleneck** — it can't parallelise across time steps, making it slower than BoE per epoch.

#### Limitations of Simple RNN (MODEL-v2)
- **Vanishing gradients**: For long sequences (>50-100 tokens), the gradient signal from early tokens vanishes during backpropagation. The RNN effectively has a "memory horizon" of ~10-20 tokens.
- **This is the #1 limitation** that motivated the invention of LSTMs and GRUs.
- **Next**: Notebook 06 empirically studies this vanishing gradient problem with gradient norm tracking and eigenvalue analysis.

#### Common Interview Questions
1. **Q: What is the hidden state in an RNN?**
   - The hidden state $h_t$ is a compressed summary of all tokens seen from position 1 to $t$. It acts as the model's "memory" and is computed recursively using the transition equation.

2. **Q: Why use `pack_padded_sequence`?**
   - Without it, the RNN processes PAD tokens through the recurrence, contaminating the hidden state. Packing tells PyTorch to skip PAD positions and return the hidden state from the last *real* token.

3. **Q: Why is the RNN slower than BoE even though it has fewer "effective" operations?**
   - The RNN has a **sequential dependency**: $h_t$ depends on $h_{t-1}$. This means the T time steps must be computed one after another, not in parallel. BoE processes all tokens simultaneously via matrix operations.


---
### Section 11: Save Experiment Results


In [ ]:
# Save results for documentation
rnn_results = {
    "experiment": "EXP-05",
    "model": "SimpleRNN (50d emb, 64h)",
    "embedding_dim": 50,
    "hidden_dim": 64,
    "param_count": sum(p.numel() for p in model.parameters()),
    "epochs": EPOCHS,
    "best_val_f1": round(best_val_f1, 4),
    "accuracy": round(metrics_rnn["accuracy"], 4),
    "precision_macro": round(metrics_rnn["precision_macro"], 4),
    "recall_macro": round(metrics_rnn["recall_macro"], 4),
    "avg_epoch_time": round(np.mean(history["epoch_time"]), 1),
    "total_time": round(sum(history["epoch_time"]), 1),
    "gradient_clipping": MAX_GRAD_NORM,
}

with open("../outputs/rnn_results.json", "w") as f:
    json.dump(rnn_results, f, indent=2)

print("Saved: outputs/rnn_best.pt")
print("Saved: outputs/rnn_results.json")
print("Saved: outputs/rnn_curves.png")
print("Saved: outputs/rnn_confusion.png")
print("Saved: outputs/boe_vs_rnn_comparison.png")
print("\nPhase 5 notebook 05 complete! ✅")
print("\nNext: Run 06_vanishing_gradient_study.ipynb for gradient analysis.")
